In [ ]:
import json
from uuid import uuid4
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline
from huggingface_hub import snapshot_download
import chromadb
from tqdm import tqdm
import requests
import pandas as pd
from rag_bench import evaluator, data
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from .embedder import insert_dataset

In [3]:
HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"

PUBLIC_TEXTS_REPO: str = "ai-forever/hist-rag-bench-public-texts"
PUBLIC_QA_REPO: str = "ai-forever/hist-rag-bench-public-question"


def get_private_qa_dataset(version):
    return load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version)


def get_private_texts_dataset(version):
    return load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version)

# map public_id:private_id
def get_public_to_private_texts_mapping(version):
    private_texts_ds = get_private_texts_dataset(version)
    mapping = {}
    for item in private_texts_ds["train"]:
        mapping[item["public_id"]] = item["id"]
    return mapping


texts_ds, questions_ds, version = data.get_datasets(is_hist=True)
texts_ds = get_private_texts_dataset(version)
qa_dataset = get_private_qa_dataset(version)
mapping = get_public_to_private_texts_mapping(version)
texts_ds, questions_ds, qa_dataset

Latest texts version: 1.15.0
Latest questions version: 1.15.0
Loaded texts dataset with 542 texts
Loaded questions dataset with 600 questions


(DatasetDict({
     train: Dataset({
         features: ['id', 'public_id', 'text'],
         num_rows: 542
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['id', 'question'],
         num_rows: 600
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['id', 'public_id', 'text_ids', 'text', 'url', 'question', 'answer', 'type'],
         num_rows: 600
     })
 }))

In [4]:
COLLECTION_NAME: str = "dragon_benchmark"
CHROMA_PATH: str = "./chroma_db"


chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = None

try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception as e:
    print(f"Коллекция {COLLECTION_NAME} уже существует и будет перезатерта: {e}")
finally:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            "hnsw:space": "cosine",
        }
    )
    print(f"Коллекция {COLLECTION_NAME} создана по пути {CHROMA_PATH}")

Коллекция dragon_benchmark создана по пути ./chroma_db


In [ ]:
model = SentenceTransformer('intfloat/multilingual-e5-small')

insert_dataset(dataset=texts_ds, collection=collection, model=model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 517.56it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Added batch 1, size: 1000
Added batch 2, size: 1000
Added batch 3, size: 370
Всего добавлено 2370 чанков в коллекцию dragon_benchmark


In [ ]:
from Dragon.generation import LLMGenerator
from Dragon.retriever import ChromaRetriever

private_to_public = {v: k for k, v in mapping.items()}
def rrag_pipeline(query, collection):
    results = ChromaRetriever(collection).top_n(query=question, n=5)
    found_private_ids = [int(x["id"]) for x in results["metadatas"][0]]
    found_ids = [private_to_public[pid] for pid in found_private_ids if pid in private_to_public]
    relevant_chunks = results['documents'][0]
    context = "\n\n---\n\n".join(relevant_chunks)
    
    answer = LLMGenerator().generate(context=context, query=query)
    return found_ids, answer


results = {}
public_questions = [{"id": str(row["id"]), "question": row["question"]} for row in questions_ds["train"]]

for q in tqdm(public_questions, desc="Обработка вопросов"):
    q_id = q["id"]
    question = q["question"]
    try:
        found_ids, model_answer = rrag_pipeline(question, collection)
        results[q_id] = {
            "found_ids": found_ids,
            "model_answer": model_answer,
            "status": "ok"
        }
    except Exception as e:
        print(f"Ошибка при обработке вопроса {q_id}: {e}")
        results[q_id] = {
            "found_ids": [],
            "model_answer": None,
            "status": "error",
            "error": str(e)
        }

with open("dragon_chroma_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ Результаты сохранены: dragon_chroma_results.json")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 529.54it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Обработка вопросов: 100%|██████████| 600/600 [38:04<00:00,  3.81s/it]  

✅ Результаты сохранены: dragon_chroma_results.json


In [7]:
ev_res = evaluator.evaluate_rag_results(results, qa_dataset, mapping, evaluator.RAGEvaluator())

In [13]:
ev_res["average_metrics"]

{'retrieval': {'hit_rate': np.float64(0.91),
  'mrr': np.float64(0.8297222222222222),
  'precision': np.float64(0.3726666666666667)},
 'generation': {'rouge1': np.float64(0.4013752247956151),
  'rougeL': np.float64(0.38130987686322576),
  'substr_match': np.float64(0.44)}}

In [11]:
ev_res["aggregated_metrics"]

{'simple': {'retrieval': {'hit_rate': np.float64(0.9133333333333333),
   'mrr': np.float64(0.8407777777777778),
   'precision': np.float64(0.38399999999999995)},
  'generation': {'rouge1': np.float64(0.3926312406266652),
   'rougeL': np.float64(0.38972927984235145),
   'substr_match': np.float64(0.5066666666666667)}},
 'cond': {'retrieval': {'hit_rate': np.float64(0.9333333333333333),
   'mrr': np.float64(0.8585555555555555),
   'precision': np.float64(0.396)},
  'generation': {'rouge1': np.float64(0.5421286021534198),
   'rougeL': np.float64(0.5404619354867531),
   'substr_match': np.float64(0.7266666666666667)}},
 'set': {'retrieval': {'hit_rate': np.float64(0.8933333333333333),
   'mrr': np.float64(0.814),
   'precision': np.float64(0.38266666666666665)},
  'generation': {'rouge1': np.float64(0.2914674487648484),
   'rougeL': np.float64(0.21577468448627138),
   'substr_match': np.float64(0.0)}},
 'mh': {'retrieval': {'hit_rate': np.float64(0.9),
   'mrr': np.float64(0.80555555555555

In [10]:
rrag_pipeline("Какой автомобиль, имеющий аналогичные размеры с Geely Monjaro, планируется к выпуску весной следующего года?", collection)

([179, 179, 397, 508, 435],
 'Из контекста не следует, какой автомобиль планируется к выпуску весной следующего года и имеет аналогичные размеры с Geely Monjaro. Ответа на этот вопрос в предоставленном тексте нет.')